# Freight Rate Prediction — interactive walkthrough

Machine Learning Engineer assessment, end-to-end demo. Runs the exact code that produced the
submitted files, so nothing here is a re-implementation.

**What happens in this notebook:** clone the repo → explore the data → quantify the data-quality
issues → run rolling-origin CV → run the official `score.py` → inspect both submission files.

Runtime on a free Colab CPU: about 3 minutes. Runtime → Run all.

## 0. Setup

The repository carries the supplied data, so cloning it is the whole setup step.

In [ ]:
REPO = "https://github.com/fajarsetiawan17/freight-rate-ml.git"

import os, subprocess, sys

if not os.path.isdir("/content/freight-rate-ml"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/freight-rate-ml"], check=True)

os.chdir("/content/freight-rate-ml")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "lightgbm==4.7.0", "scikit-learn", "matplotlib"], check=True)

print("cwd      :", os.getcwd())
print("data     :", sorted(os.listdir("data")))

## 1. What the data looks like

The single most important fact about this dataset: **posted rate is almost a pure function of
distance**. Correlation of `log(rate)` with `log(distance)` is 0.99 once the impossible price rows
are removed — and 0.967 even with them in. Everything else the model does is a level correction on
top of that relationship.

In [ ]:
import numpy as np, pandas as pd

train = pd.read_csv("data/train_test.csv", parse_dates=["date"])
valid = pd.read_csv("data/validation.csv", parse_dates=["date"])
december = pd.read_csv("data/december_chart_inputs.csv", parse_dates=["date"])

print(f"train    {len(train):,} rows | {train.date.min():%Y-%m-%d} -> {train.date.max():%Y-%m-%d}")
print(f"valid    {len(valid):,} rows | {valid.date.min():%Y-%m-%d} -> {valid.date.max():%Y-%m-%d}")
print(f"december {len(december):,} rows")
print()
print("columns:", list(train.columns))
print()
corr = np.corrcoef(np.log(train.distance), np.log(train.posted_rate))[0, 1]
print(f"corr(log distance, log rate), all rows       = {corr:.4f}")

# once the impossible $/mile rows are removed the relationship is near-deterministic
rpm_all = train.posted_rate / train.distance
clean = train[(rpm_all >= 1) & (rpm_all <= 6)]
corr_clean = np.corrcoef(np.log(clean.distance), np.log(clean.posted_rate))[0, 1]
print(f"corr(log distance, log rate), valid rows     = {corr_clean:.4f}   "
      f"(n={len(clean):,})   <- the 0.99 quoted in the report")
print(f"rate skewness                                = {train.posted_rate.skew():.2f}   <- hence the log target")

### Rate per mile

Plotting dollars-per-mile against distance makes the pricing structure visible — and exposes the
rows that cannot be real prices.

In [ ]:
import matplotlib.pyplot as plt

rpm = train.posted_rate / train.distance
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].scatter(train.distance, train.posted_rate, s=1, alpha=.15)
ax[0].set(xlabel="distance (mi)", ylabel="posted rate ($)", title="rate vs distance")
ax[1].hist(rpm.clip(0, 12), bins=120, color="#3b6ea5")
ax[1].set(xlabel="$ per mile (clipped at 12)", ylabel="loads", title="rate-per-mile distribution")
plt.tight_layout(); plt.show()

print(f"$/mile  min {rpm.min():.2f} | p1 {rpm.quantile(.01):.2f} | median {rpm.median():.2f} "
      f"| p99 {rpm.quantile(.99):.2f} | max {rpm.max():.2f}")

## 2. Data-quality issues

Six defects, each quantified below with the handling that `pipeline.py` applies. A $/mile outside
1–6 is a label error, not a price — those rows are dropped from **training only**, and the loads
still receive predictions.

In [ ]:
def report_quality(df, name):
    print(f"--- {name} ({len(df):,} rows)")
    if "posted_rate" in df:
        rpm = df.posted_rate / df.distance
        print(f"  impossible $/mile (<1 or >6)   : {((rpm < 1) | (rpm > 6)).sum():>6,}")
    print(f"  negative weight (sign flip)    : {(df.weight < 0).sum():>6,}"
          f"   min {df.weight.min():,.0f} lb")
    print(f"  missing weight                 : {df.weight.isna().sum():>6,}")
    print(f"  missing market_index           : {df.market_index.isna().sum():>6,}")
    dup = df.duplicated(["pickup", "delivery", "equipment", "date"], keep="first").sum()
    print(f"  duplicate route+date+equip     : {dup:>6,}   (separate loads, kept)")

report_quality(train, "train")
report_quality(valid, "validation")
print()
print(f"december file columns           : {list(december.columns)}")
print("  no pickup_lat / pickup_lon in this file  <- 64-city lookup rebuilt from train")

## 3. Training and validation

Validation is **rolling-origin**: three expanding-window folds, each training up to the end of one
month and predicting the next, unseen month.

Random K-fold would be wrong here. The real submission is a two-month forward forecast, so a random
split leaks future information into training and makes the score look better than it is.

Expect roughly: `MAE 53.56 / 56.38 / 51.93`, `MAPE ~2.1%`.

In [ ]:
!python pipeline.py --cv

### Why `market_index` and `quote_signal` are excluded

Both are level-shifted between the development window and the prediction window, so a tree model
learns their relationship and then extrapolates a price level the market is not paying. The ablation
measures the damage rather than assuming it.

In [ ]:
print(pd.read_csv("outputs/ablation.csv").to_string(index=False))

## 4. The official scorer

`score.py` is supplied by the assessment and is **not modified**. It validates both submission files
and renders the fixed December chart.

In [ ]:
!python score.py --predictions outputs/validation_predictions.csv --december-predictions outputs/december_chart_inputs.csv

### December chart

The one artifact the brief asks for by name.

In [ ]:
from IPython.display import Image
Image("scorer_results/candidate_december.png")

## 5. Submission files

Two outputs. `validation_predictions.csv` keeps exactly `load_id,predicted_rate`. The December file
keeps the original seven columns **in their original order** plus `predicted_rate` — the scorer
rejects any other shape.

In [ ]:
preds = pd.read_csv("outputs/validation_predictions.csv")
dec = pd.read_csv("outputs/december_chart_inputs.csv")

print("validation_predictions.csv")
print(f"  shape   {preds.shape} | columns {list(preds.columns)}")
print(f"  $  min {preds.predicted_rate.min():,.2f} | median {preds.predicted_rate.median():,.2f}"
      f" | max {preds.predicted_rate.max():,.2f}")
print()
print("december_chart_inputs.csv")
print(f"  shape   {dec.shape} | columns {list(dec.columns)}")
print(f"  $  min {dec.predicted_rate.min():,.2f} | mean {dec.predicted_rate.mean():,.2f}"
      f" | max {dec.predicted_rate.max():,.2f}")
print()
preds.head()

### December, by day of week

Mid-week peaks, Sunday troughs — the weekly shape the chart above shows.

In [ ]:
by_dow = (dec.assign(dow=pd.to_datetime(dec.date).dt.day_name())
             .groupby("dow", sort=False).predicted_rate.mean()
             .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]))
print(by_dow.round(2).to_string())

## What this notebook does not do

No report rendering. `build_report.py` and `make_report_assets.py` need a Word/LibreOffice step for
the PDF, which is why the rendered report ships in the repo as `report/Freight_Rate_ML_Report.pdf`
instead of being regenerated here.

Everything else — cleaning, features, CV, both submission files, and the scorer — runs above
unmodified from the submitted source.